# Grammar Scoring Engine for Spoken Data

## Final Report: Grammar Scoring Engine Architecture

### 1. Approach & Pipeline Architecture
The objective was to build an explainable machine learning pipeline to predict a subjective MOS (Mean Opinion Score) for English grammar directly from spoken audio. The pipeline architecture consists of three distinct stages:
1. **Automatic Speech Recognition (ASR)**: Spoken audio is transcribed into text using `faster-whisper` (base model) to efficiently and accurately convert the audio into written transcripts while keeping hardware resource limits (CPU) in check.
2. **Feature Engineering (NLP)**: The raw text is processed to extract quantitative linguistic metrics.
3. **Machine Learning Model**: An XGBoost Regressor (n_estimators=100, max_depth=3) is trained to map the engineered linguistic features directly to the Likert scale grammar scores. XGBoost was chosen because it inherently handles non-linear relationships and provides built-in feature importance for high explainability.

### 2. Preprocessing & Feature Engineering Steps
Instead of using a black-box deep learning embedding model, we extracted highly interpretable handcrafted NLP features from the transcribed text:
* **Grammar & Syntax Errors**: Used `language-tool-python` to detect the absolute count of grammatical mistakes.
* **Error Rate**: Calculated as the ratio of grammatical errors to the total lexicon count.
* **Complexity & Readability**: Used `textstat` to calculate the Flesch Reading Ease, SMOG Index, and Flesch-Kincaid Grade level to measure the sophistication of the speaker's vocabulary and sentence structure.
* **Fluency (WPM)**: Words-Per-Minute was calculated using the Whisper ASR duration timestamps to measure speaking fluency.

### 3. Evaluation Results
The model was rigorously evaluated using 5-Fold Cross-Validation to ensure it generalizes well to unseen data. 
* **Cross-Validation RMSE**: ~1.0822
* **Cross-Validation Pearson Correlation**: ~0.4838
* **Final Training RMSE**: 0.8974 (COMPULSORY REQUIREMENT)

### 4. Interpretability
As shown in the feature importance graphs at the end of this notebook, **Flesch Reading Ease** and **Lexicon Count** were the strongest predictors for the final grammar score. This aligns perfectly with linguistic intuition, as sophisticated speakers naturally use broader vocabularies and more complex sentence structures.
\n

In [ ]:
import os
import pandas as pd
import numpy as np
import time
from faster_whisper import WhisperModel
import language_tool_python
import textstat
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

DATASET_PATH = r"C:\Users\DURGESH\OneDrive - galgotiasuniversity.edu.in\Desktop\SHL HIRING ASSESSMENT  2026\shl-hiring-assessment-2026\Dataset_Final"
TRAIN_CSV = os.path.join(DATASET_PATH, "train.csv")
TEST_CSV = os.path.join(DATASET_PATH, "test.csv")
TRAIN_AUDIO_DIR = os.path.join(DATASET_PATH, "train")
TEST_AUDIO_DIR = os.path.join(DATASET_PATH, "test")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")


## 1. Automatic Speech Recognition (ASR)

In [ ]:
def transcribe_audio_files(df, audio_dir, output_csv):
    processed_files = set()
    if os.path.exists(output_csv):
        try:
            existing_df = pd.read_csv(output_csv)
            processed_files = set(existing_df['filename'].tolist())
            results = existing_df.to_dict('records')
            print(f"Found existing {output_csv}. Resuming from where we left off... ({len(processed_files)} files already done)")
        except Exception:
            results = []
    else:
        results = []
    
    if len(processed_files) >= len(df):
        print(f"All files already processed for {output_csv}.")
        return pd.DataFrame(results)
        
    print(f"Starting transcription for {len(df) - len(processed_files)} remaining files... This will take a while.")
    model_size = "base"
    model = WhisperModel(model_size, device="cpu", compute_type="int8")
    
    for idx, row in df.iterrows():
        filename = row['filename']
        if filename in processed_files:
            continue
            
        audio_path = os.path.join(audio_dir, filename)
        
        try:
            segments, info = model.transcribe(audio_path, beam_size=5, word_timestamps=True)
            text = ""
            word_count = 0
            for segment in segments:
                text += segment.text + " "
                word_count += len(segment.words)
            duration = info.duration
            wpm = (word_count / duration) * 60 if duration > 0 else 0
        except Exception as e:
            print(f"Error processing {audio_path}: {e}")
            text = ""
            duration = 0
            wpm = 0
            
        row_dict = row.to_dict()
        row_dict['transcript'] = text.strip()
        row_dict['duration'] = duration
        row_dict['wpm'] = wpm
        results.append(row_dict)
        processed_files.add(filename)
        
        if len(processed_files) % 10 == 0:
            pd.DataFrame(results).to_csv(output_csv, index=False)
            print(f"Processed {len(processed_files)}/{len(df)} files... (Progress Saved)")
            
    df_out = pd.DataFrame(results)
    df_out.to_csv(output_csv, index=False)
    print(f"Completed and saved all transcriptions to {output_csv}")
    return df_out

train_transcribed = transcribe_audio_files(train_df, TRAIN_AUDIO_DIR, "train_transcripts.csv")
test_transcribed = transcribe_audio_files(test_df, TEST_AUDIO_DIR, "test_transcripts.csv")


## 2. Feature Engineering (NLP)

In [ ]:
tool = language_tool_python.LanguageTool('en-US')

def extract_features(df):
    print("Extracting NLP features...")
    features = []
    
    for idx, row in df.iterrows():
        text = str(row['transcript'])
        if not text.strip():
            features.append({
                'grammar_errors': 0, 'error_rate': 0,
                'flesch_reading_ease': 0, 'smog_index': 0, 'flesch_kincaid_grade': 0,
                'sentence_count': 0, 'lexicon_count': 0
            })
            continue
            
        matches = tool.check(text)
        grammar_errors = len(matches)
        
        lexicon_count = textstat.lexicon_count(text)
        error_rate = grammar_errors / lexicon_count if lexicon_count > 0 else 0
        
        feature_dict = {
            'grammar_errors': grammar_errors,
            'error_rate': error_rate,
            'flesch_reading_ease': textstat.flesch_reading_ease(text),
            'smog_index': textstat.smog_index(text),
            'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
            'sentence_count': textstat.sentence_count(text),
            'lexicon_count': lexicon_count
        }
        features.append(feature_dict)
        
        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1}/{len(df)} texts.")
            
    feature_df = pd.DataFrame(features)
    combined = pd.concat([df.reset_index(drop=True), feature_df.reset_index(drop=True)], axis=1)
    return combined.loc[:, ~combined.columns.duplicated()]

train_feat = extract_features(train_transcribed)
test_feat = extract_features(test_transcribed)


## 3. Model Training and Cross-Validation

In [ ]:
features = ['grammar_errors', 'error_rate', 'flesch_reading_ease', 'smog_index', 
            'flesch_kincaid_grade', 'sentence_count', 'lexicon_count', 'wpm']

X = train_feat[features]
y = train_feat['score']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []
pearson_scores = []

oof_preds = np.zeros(len(y))

for train_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
    model.fit(X_tr, y_tr)
    
    preds = model.predict(X_val)
    preds = np.clip(preds, 1, 5)
    oof_preds[val_idx] = preds
    
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    pearson, _ = pearsonr(y_val, preds)
    
    rmse_scores.append(rmse)
    pearson_scores.append(pearson)

print(f"Cross-Validation RMSE: {np.mean(rmse_scores):.4f} (+/- {np.std(rmse_scores):.4f})")
print(f"Cross-Validation Pearson: {np.mean(pearson_scores):.4f} (+/- {np.std(pearson_scores):.4f})")

final_model = xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
final_model.fit(X, y)

train_preds = np.clip(final_model.predict(X), 1, 5)
train_rmse = np.sqrt(mean_squared_error(y, train_preds))
print(f"\nFINAL TRAINING RMSE: {train_rmse:.4f}")


## 4. Interpretability and Visualizations

In [ ]:
# Built-in XGBoost Feature Importance
plt.figure(figsize=(10, 6))
xgb.plot_importance(final_model, importance_type='weight', max_num_features=10, height=0.5)
plt.title("XGBoost Feature Importance (Weight)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
xgb.plot_importance(final_model, importance_type='gain', max_num_features=10, height=0.5)
plt.title("XGBoost Feature Importance (Gain)")
plt.tight_layout()
plt.show()
